In [5]:
import pandas as pd

# Paths
ARXIV_PATH   = r"c:\Users\hp\Desktop\Research-System\training\datasets\dataset_arxiv.csv"
PUBMED_PATH  = r"c:\Users\hp\Desktop\Research-System\training\datasets\dataset_pubmed.csv"
OPENALEX_PATH = r"c:\Users\hp\Desktop\Research-System\training\datasets\dataset_openalex.csv"
COMBINED_PATH = r"c:\Users\hp\Desktop\Research-System\training\datasets\dataset_combined.csv"

In [6]:
# Load
arxiv   = pd.read_csv(ARXIV_PATH)
pubmed  = pd.read_csv(PUBMED_PATH)
openalex = pd.read_csv(OPENALEX_PATH)

# Quick sanity check – all three should have the same columns
assert list(arxiv.columns) == list(pubmed.columns) == list(openalex.columns), "Column mismatch!"

print("Before merge:")
print(f"  arXiv:    {len(arxiv):,}")
print(f"  PubMed:   {len(pubmed):,}")
print(f"  OpenAlex: {len(openalex):,}")

Before merge:
  arXiv:    394,103
  PubMed:   119,997
  OpenAlex: 339,757


In [7]:
# Combine
df = pd.concat([arxiv, pubmed, openalex], ignore_index=True)

# Drop explicit source column since we don’t need it for training
df = df.drop(columns=["source"])

# Cross‑source deduplication (title + abstract)
before = len(df)
df = df.drop_duplicates(subset=["title", "abstract"]).reset_index(drop=True)
print(f"\nCross‑source duplicates removed: {before - len(df):,}")

# Final shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
df.to_csv(COMBINED_PATH, index=False)
print(f"\nCombined dataset saved to: {COMBINED_PATH}")
print(f"Total papers: {len(df):,}")

# Quick distribution check
print("\nMain category distribution:")
print(df["main_label"].value_counts().to_string())
print("\nSubcategory distribution:")
print(df["sub_label"].value_counts().to_string())


Cross‑source duplicates removed: 89

Combined dataset saved to: c:\Users\hp\Desktop\Research-System\training\datasets\dataset_combined.csv
Total papers: 853,768

Main category distribution:
main_label
Chemistry               134061
Computer Science        119998
Biology & Medicine      119997
Mathematics             119991
Physics                 119988
Engineering             119904
Economics & Business    119829

Subcategory distribution:
sub_label
Natural Language Processing               20000
Neuroscience                              20000
Astrophysics                              20000
Analysis                                  20000
Artificial Intelligence                   20000
High Energy Physics                       20000
Immunology & Microbiology                 20000
Condensed Matter                          20000
Systems & Control                         20000
Machine Learning                          20000
Genetics & Molecular Biology              20000
Combinatorics   

In [8]:
df = pd.read_csv(COMBINED_PATH)

print(f"Rows before removal: {len(df):,}")

# Remove the rogue category
df = df[df["sub_label"] != "Physical Chemistry"].reset_index(drop=True)

print(f"Rows after removal:  {len(df):,}")

# Quick sanity check
assert df[df["sub_label"] == "Physical Chemistry"].empty, "Failed to remove Physical Chemistry!"

# Save the cleaned version
df.to_csv(COMBINED_PATH, index=False)
print("\nCleaned dataset saved to:", COMBINED_PATH)

# Show updated distribution for Chemistry
print("\nChemistry subcategory counts after fix:")
print(df[df["main_label"] == "Chemistry"]["sub_label"].value_counts().to_string())

Rows before removal: 853,768
Rows after removal:  839,642

Cleaned dataset saved to: c:\Users\hp\Desktop\Research-System\training\datasets\dataset_combined.csv

Chemistry subcategory counts after fix:
sub_label
Inorganic Chemistry                   19998
Organic Chemistry                     19997
Spectroscopy                          19988
Analytical Chemistry                  19986
Physical and Theoretical Chemistry    19983
Electrochemistry                      19983


In [9]:
df = pd.read_csv(COMBINED_PATH)

print("=" * 60)
print("FINAL DATASET DISTRIBUTION")
print("=" * 60)
print(f"Total papers: {len(df):,}\n")

# Iterate over main categories sorted alphabetically
for main_cat in sorted(df["main_label"].unique()):
    sub_df = df[df["main_label"] == main_cat]
    print(f"{main_cat} ({len(sub_df):,} papers)")
    print("-" * 40)
    for sub, count in sub_df["sub_label"].value_counts().items():
        print(f"  {sub:<45} {count:>6,}")
    print()

FINAL DATASET DISTRIBUTION
Total papers: 839,642

Biology & Medicine (119,997 papers)
----------------------------------------
  Neuroscience                                  20,000
  Genetics & Molecular Biology                  20,000
  Immunology & Microbiology                     20,000
  Cardiology                                    20,000
  Pharmacology                                  19,999
  Oncology                                      19,998

Chemistry (119,935 papers)
----------------------------------------
  Inorganic Chemistry                           19,998
  Organic Chemistry                             19,997
  Spectroscopy                                  19,988
  Analytical Chemistry                          19,986
  Physical and Theoretical Chemistry            19,983
  Electrochemistry                              19,983

Computer Science (119,998 papers)
----------------------------------------
  Natural Language Processing                   20,000
  Machine Lea